# Amazon Bedrock AgentCore Policy - Automated Policy Creation

## Overview

This notebook demonstrates **automated, agent-driven policy lifecycle creation** for Amazon Bedrock AgentCore.
Rather than manually writing Cedar policies, a policy orchestration agent inspects the tools attached to an
AgentCore Gateway, reads the RBAC permission manifests those tools emit, auto-generates Cedar policies via
the NL2Cedar API, routes each policy through a human reviewer (SNS notification + interactive approval),
and finally commits approved policies to the Policy Engine.

### What you will learn

1. How tools attached to an AgentCore Gateway emit RBAC permission requirements
2. How to convert RBAC metadata into natural language policy statements
3. How to use the NL2Cedar API to auto-generate valid Cedar policies
4. How to implement a human-in-the-loop review step using Amazon SNS
5. How to programmatically create policies in a Policy Engine and enforce them

---

## Architecture

```
  Tools (Lambda)                     Policy Orchestration Agent
       |                                        |
       | register with schema                   |
       v                           1. list_gateway_tools
  AgentCore Gateway  <----------   2. get_tool_rbac_permissions
  (emits RBAC manifest)            3. generate_nl_policy_statement
       |                           4. generate_cedar_policy (NL2Cedar)
       |                           5. request_human_approval (SNS)
       |                           6. create_policy_in_engine
       |                                        |
       v                                        v
  Policy Engine  <------- approved Cedar policies
  (enforces rules)         Human Reviewer
       |                   (email via SNS)
       | ALLOW / DENY
       v
  Tool Invocation
```

---

## Demo Scenario: Financial Data Platform

Two tools are deployed on the gateway:

| Tool | Function | RBAC Requirements |
|------|----------|-------------------|
| FinancialReportTarget | `get_financial_report` | Roles: analyst/senior-analyst/manager; classification_level must be `internal`; region in US/EU/APAC |
| TradeExecutionTarget  | `execute_trade`        | Roles: trader/portfolio-manager; amount ≤ $500,000 |

---

## Prerequisites

- AWS CLI configured with appropriate credentials
- Python 3.10+ with boto3
- `bedrock_agentcore_starter_toolkit` installed
- An email address to receive human review notifications (SNS)

---
# Step 0: Environment Setup

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sys
import json
import logging
import time
from pathlib import Path

import boto3

# Add scripts directory to path so helper modules are importable
scripts_dir = Path.cwd() / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

session = boto3.Session()
region = session.region_name or "us-east-1"

try:
    sts = session.client("sts")
    identity = sts.get_caller_identity()
    account_id = identity["Account"]
    print("AWS credentials verified")
    print(f"  Account : {account_id}")
    print(f"  Identity: {identity['Arn']}")
    print(f"  Region  : {region}")
except Exception as e:
    print(f"AWS credentials error: {e}")
    print("Please configure AWS CLI with: aws configure")

---
# Step 1: Deploy Infrastructure

This step deploys two Lambda functions (FinancialReportTool, TradeExecutionTool),
creates an AgentCore Gateway, and attaches the Lambda functions as targets.

When each tool is attached to the Gateway it **emits an RBAC permission manifest**
— a structured description of the access control requirements that the policy
orchestrator will later use to generate Cedar policies automatically.

### RBAC Manifest Format

Each manifest entry contains:
- `allowed_roles`: Which user roles may call this tool
- `data_classification`: Sensitivity level (internal, restricted, public)
- `constraints`: Input parameter constraints (allowed values, max values)
- `requires_approval`: Whether high-value operations need extra sign-off

In [ ]:
# Preview the RBAC manifest that will be emitted by each tool
from setup_infrastructure import TOOL_RBAC_MANIFEST

print("RBAC Permission Manifest (emitted when tools are attached to Gateway)")
print("=" * 70)
for action_key, entry in TOOL_RBAC_MANIFEST.items():
    print(f"\nTool: {action_key}")
    print(f"  Description   : {entry['description']}")
    rbac = entry["rbac"]
    print(f"  Allowed Roles : {', '.join(rbac.get('allowed_roles', []))}")
    print(f"  Classification: {rbac.get('data_classification', 'N/A')}")
    print(f"  Requires Appr : {rbac.get('requires_approval', False)}")
    for param, c in rbac.get("constraints", {}).items():
        if "enum" in c:
            print(f"  Constraint    : {param} must be one of {c['enum']}")
        elif "max" in c:
            print(f"  Constraint    : {param} must not exceed {c['max']}")
print("=" * 70)

In [ ]:
# Deploy Lambda functions and create the AgentCore Gateway
from setup_infrastructure import setup_infrastructure

config = setup_infrastructure(region=region)

In [ ]:
# Load config from file (stable reference for the rest of the notebook)
with open("config.json") as f:
    config = json.load(f)

GATEWAY_ID  = config["gateway"]["gateway_id"]
GATEWAY_ARN = config["gateway"]["gateway_arn"]
GATEWAY_URL = config["gateway"]["gateway_url"]

print("Gateway configuration loaded")
print(f"  Gateway ID : {GATEWAY_ID}")
print(f"  Gateway ARN: {GATEWAY_ARN}")
print(f"  Gateway URL: {GATEWAY_URL}")
print(f"\nRBAC manifest contains {len(config['rbac_manifest'])} tool entries")

---
# Step 2: Set Up Human Review Infrastructure

Before running the orchestrator we need to create:
1. An **SNS topic** — the orchestrator publishes policy review requests here
2. An **SQS queue** subscribed to the topic — captures programmatic responses

If you provide your email address below, you will receive an email from Amazon SNS
for each policy the orchestrator generates. You must confirm the subscription by
clicking the link in the confirmation email before you will receive notifications.

The interactive approval in the notebook is the authoritative decision mechanism.
The SNS email provides an audit trail.

In [ ]:
# Enter the reviewer email address (leave blank to skip email notifications)
reviewer_email = input("Enter reviewer email address (or press Enter to skip): ").strip()
if reviewer_email:
    print(f"Reviewer email set to: {reviewer_email}")
else:
    print("No email provided - policies will only be reviewed interactively in the notebook.")

In [ ]:
from human_review import setup_review_infrastructure, subscribe_email_to_topic

print("Creating human review infrastructure...")
review_infra = setup_review_infrastructure(region=region)

SNS_TOPIC_ARN  = review_infra["sns_topic_arn"]
SQS_QUEUE_URL  = review_infra["sqs_queue_url"]

# Subscribe reviewer email if provided
if reviewer_email:
    sns_client = boto3.client("sns", region_name=region)
    subscribe_email_to_topic(sns_client, SNS_TOPIC_ARN, reviewer_email)
    print("  Check your email to confirm the SNS subscription.")

print(f"\nSNS Topic ARN : {SNS_TOPIC_ARN}")
print(f"SQS Queue URL : {SQS_QUEUE_URL}")

# Persist review infra details to config for cleanup
config["review_infra"] = review_infra
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
print("\nReview infrastructure details saved to config.json")

---
# Step 3: Create Policy Engine

A Policy Engine holds a collection of Cedar policies. It is created here but
not yet attached to the Gateway — the orchestrator will populate it with
human-approved policies first. We will attach it to the Gateway in Step 5.

Until the Policy Engine is attached, all tools on the Gateway are accessible
with no restrictions.

In [ ]:
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=region)

print("Creating Policy Engine...")
engine_resp = agentcore_client.create_policy_engine(
    name="FinancialDataPlatformPolicyEngine",
    description="Auto-managed Cedar policies for Financial Data Platform tools",
    tags={"Environment": "Demo", "ManagedBy": "PolicyOrchestrator"},
)

POLICY_ENGINE_ID  = engine_resp["policyEngineId"]
POLICY_ENGINE_ARN = engine_resp["policyEngineArn"]

print(f"Policy Engine created")
print(f"  ID : {POLICY_ENGINE_ID}")
print(f"  ARN: {POLICY_ENGINE_ARN}")

# Save to config
config["policy_engine_id"]  = POLICY_ENGINE_ID
config["policy_engine_arn"] = POLICY_ENGINE_ARN
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
print("\nPolicy engine details saved to config.json")

---
# Step 4: Run the Policy Orchestration Agent

The orchestrator is a Strands agent with six purpose-built tools:

| Tool | What it does |
|------|--------------|
| `list_gateway_tools` | Discover all tools on the Gateway via MCP |
| `get_tool_rbac_permissions` | Read the RBAC manifest emitted by a tool |
| `generate_nl_policy_statement` | Convert RBAC -> natural language |
| `generate_cedar_policy` | Call NL2Cedar to produce Cedar syntax |
| `request_human_approval` | Send SNS notification + interactive approval |
| `create_policy_in_engine` | Commit approved policy to the Policy Engine |

### What will happen

1. The agent will discover the two tools on the Gateway
2. For each tool it will read the RBAC manifest, generate a NL statement, then call NL2Cedar
3. It will display the generated Cedar policy and ask **you** to approve or reject
4. Approved policies are immediately created in the Policy Engine

> **Note**: The agent will pause twice for your interactive input — once per tool.

In [ ]:
from policy_orchestrator import configure_orchestrator, create_policy_orchestrator_agent

# Configure module-level context used by all orchestrator tools
configure_orchestrator(
    config=config,
    policy_engine_id=POLICY_ENGINE_ID,
    sns_topic_arn=SNS_TOPIC_ARN,
    region=region,
)

# Create the agent
orchestrator_agent = create_policy_orchestrator_agent(model_id="amazon.nova-lite-v1:0")
print("Policy Orchestration Agent ready")

In [ ]:
# Run the orchestrator.
# The agent will work through both tools automatically and pause for
# your input when requesting approval for each generated Cedar policy.

print("=" * 70)
print("Starting automated policy management workflow")
print("=" * 70)
print()

orchestration_result = orchestrator_agent(
    """Process all tools on the Gateway using the automated policy management workflow.
For each tool:
1. Read its RBAC permissions from the manifest
2. Generate a natural language policy statement
3. Convert it to Cedar using NL2Cedar
4. Request human approval for each generated policy
5. Create approved policies in the Policy Engine

When requesting approval, show the complete Cedar policy and wait for my decision.
After processing all tools, provide a final summary."""
)

print("\n" + "=" * 70)
print("Orchestration complete")
print("=" * 70)

In [ ]:
# Verify policies were created. Policies start in CREATING state and
# transition to ACTIVE within a few seconds.
print("Waiting for policies to become ACTIVE...")
for attempt in range(15):
    policies_resp = agentcore_client.list_policies(policyEngineId=POLICY_ENGINE_ID)
    # Note: response key is "policies" (not "items")
    policies = policies_resp.get("policies", [])
    active = [p for p in policies if p.get("status") == "ACTIVE"]
    if active:
        break
    time.sleep(3)
    print(f"  attempt {attempt + 1}: {len(policies)} found, {len(active)} active...")

print(f"\nPolicies in Policy Engine '{POLICY_ENGINE_ID}':")
print("=" * 70)
if not policies:
    print("No policies created (all were rejected or an error occurred)")
else:
    for p in policies:
        print(f"  Policy ID: {p.get('policyId')}")
        print(f"  Name     : {p.get('name')}")
        print(f"  Status   : {p.get('status')}")
        cedar = p.get("definition", {}).get("cedar", {}).get("statement", "")
        print(f"  Cedar    : {cedar[:140]}...")
        print()
print("=" * 70)

---
# Step 5: Attach Policy Engine to Gateway

Now we attach the populated Policy Engine to the Gateway in **ENFORCE** mode.

From this point:
- The Gateway evaluates every inbound tool request against the Cedar policies
- The default action is **DENY** — requests are blocked unless a policy explicitly permits them
- Only tool calls matching an approved Cedar policy will be forwarded to the Lambda target

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

gw_client = GatewayClient(region_name=region)
gw_client.logger.setLevel(logging.WARNING)

print("Attaching Policy Engine to Gateway in ENFORCE mode...")
gw_client.update_gateway_policy_engine(
    gateway_identifier=GATEWAY_ID,
    policy_engine_arn=POLICY_ENGINE_ARN,
    mode="ENFORCE",
)
print(f"  Policy Engine {POLICY_ENGINE_ID} attached to Gateway {GATEWAY_ID}")
print("  Mode: ENFORCE")
print()
print("All tool calls through this Gateway are now governed by Cedar policies.")
print("Default action: DENY. Only explicitly permitted requests will succeed.")

---
# Step 6: Test the Automated Policy Enforcement

We will test the enforcement using a Strands agent that invokes the tools through
the Gateway. The tests below map directly to the Cedar policies the orchestrator created.

**FinancialReportTarget policy** — generated from RBAC:
- classification_level must be `internal`
- region must be one of `US`, `EU`, `APAC`

**TradeExecutionTarget policy** — generated from RBAC:
- amount must not exceed 500,000

### Test matrix

| Test | Tool | Parameters | Expected |
|------|------|-----------|----------|
| 1 | get_financial_report | classification=internal, region=US | ALLOW |
| 2 | get_financial_report | classification=restricted, region=US | DENY |
| 3 | get_financial_report | classification=internal, region=LATAM | DENY |
| 4 | execute_trade | amount=250000 | ALLOW |
| 5 | execute_trade | amount=750000 | DENY |

In [ ]:
import os
import requests as http_requests
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client


def get_access_token(config: dict) -> str:
    ci = config["gateway"]["client_info"]
    resp = http_requests.post(
        ci["token_endpoint"],
        data=(
            f"grant_type=client_credentials"
            f"&client_id={ci['client_id']}"
            f"&client_secret={ci['client_secret']}"
        ),
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


os.environ["AWS_DEFAULT_REGION"] = region
access_token = get_access_token(config)
print("Access token obtained")

bedrock_model = BedrockModel(model_id="amazon.nova-lite-v1:0", streaming=True)
mcp_transport = lambda: streamablehttp_client(
    GATEWAY_URL, headers={"Authorization": f"Bearer {access_token}"}
)
mcp_client = MCPClient(mcp_transport)
mcp_client.__enter__()
tools = mcp_client.list_tools_sync()

test_agent = Agent(
    model=bedrock_model,
    tools=tools,
    system_prompt="""
You are a financial platform assistant. Use only the tools provided.
Show the tool name, parameters, and result for each call.
If a call fails, clearly state that it was denied.
""",
)
print(f"Test agent ready. Tools available: {[t.tool_name for t in tools]}")

In [ ]:
# -----------------------------------------------------------------------
# Test 1: ALLOW - get_financial_report with valid parameters
# classification_level=internal (allowed), region=US (allowed)
# -----------------------------------------------------------------------
print("=" * 70)
print("TEST 1: ALLOW - get_financial_report with internal classification, US region")
print("=" * 70)
result1 = test_agent(
    "Get a quarterly financial report for the US region with internal classification level."
)
print(result1)

In [ ]:
# -----------------------------------------------------------------------
# Test 2: DENY - get_financial_report with disallowed classification
# classification_level=restricted (not in Cedar policy allowlist)
# -----------------------------------------------------------------------
print("=" * 70)
print("TEST 2: DENY - get_financial_report with restricted classification")
print("=" * 70)
result2 = test_agent(
    "Get a quarterly financial report for the US region with restricted classification level."
)
print(result2)

In [ ]:
# -----------------------------------------------------------------------
# Test 3: DENY - get_financial_report with disallowed region
# region=LATAM (not in Cedar policy allowlist)
# -----------------------------------------------------------------------
print("=" * 70)
print("TEST 3: DENY - get_financial_report for LATAM region (not in allowlist)")
print("=" * 70)
result3 = test_agent(
    "Get an annual financial report for the LATAM region with internal classification level."
)
print(result3)

In [ ]:
# -----------------------------------------------------------------------
# Test 4: ALLOW - execute_trade within the $500K limit
# amount=250000 (under limit)
# -----------------------------------------------------------------------
print("=" * 70)
print("TEST 4: ALLOW - execute_trade for $250,000 (within $500K limit)")
print("=" * 70)
result4 = test_agent(
    "Execute a buy trade for AMZN stock worth $250,000."
)
print(result4)

In [ ]:
# -----------------------------------------------------------------------
# Test 5: DENY - execute_trade exceeding the $500K limit
# amount=750000 (over limit)
# -----------------------------------------------------------------------
print("=" * 70)
print("TEST 5: DENY - execute_trade for $750,000 (exceeds $500K limit)")
print("=" * 70)
result5 = test_agent(
    "Execute a buy trade for MSFT stock worth $750,000."
)
print(result5)

# Clean up test agent MCP connection
mcp_client.__exit__(None, None, None)

In [ ]:
# Print test summary
print()
print("=" * 70)
print("TEST SUMMARY")
print("=" * 70)
print("Test 1: get_financial_report (internal / US)          -> Expected: ALLOW")
print("Test 2: get_financial_report (restricted / US)        -> Expected: DENY")
print("Test 3: get_financial_report (internal / LATAM)       -> Expected: DENY")
print("Test 4: execute_trade $250K                           -> Expected: ALLOW")
print("Test 5: execute_trade $750K                           -> Expected: DENY")
print("=" * 70)
print()
print("Cedar policies generated by the orchestrator are now enforcing")
print("the RBAC requirements that were emitted when the tools were")
print("registered on the Gateway.")

---
# Cleanup

The cleanup sequence is:
1. Detach the Policy Engine from the Gateway
2. Delete all policies in the Policy Engine
3. Delete the Policy Engine
4. Delete the SNS topic and SQS queue
5. Delete the Gateway (and Cognito resources)
6. Delete the Lambda functions
7. Delete the IAM role

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient
from human_review import cleanup_review_infrastructure

with open("config.json") as f:
    config = json.load(f)

# -----------------------------------------------------------------------
# 1. Detach Policy Engine from Gateway
# -----------------------------------------------------------------------
print("Step 1: Detaching Policy Engine from Gateway...")
try:
    gw_client = GatewayClient(region_name=config["region"])
    gw_client.logger.setLevel(logging.WARNING)
    # Read current gateway config for required fields, then re-issue the update
    # without policyEngineConfiguration to detach the policy engine.
    gw_id = config["gateway"]["gateway_id"]
    gw_resp = agentcore_client.get_gateway(gatewayIdentifier=gw_id)
    update_params = {
        "gatewayIdentifier": gw_id,
        "name": gw_resp["name"],
        "roleArn": gw_resp["roleArn"],
        "protocolType": gw_resp["protocolType"],
        "authorizerType": gw_resp["authorizerType"],
    }
    for field in ("authorizerConfiguration", "protocolConfiguration", "description"):
        if field in gw_resp:
            update_params[field] = gw_resp[field]
    agentcore_client.update_gateway(**update_params)
    print("  Policy Engine detached")
except Exception as e:
    print(f"  Warning: {e}")

# -----------------------------------------------------------------------
# 2 & 3. Delete policies and Policy Engine
# -----------------------------------------------------------------------
print("\nStep 2/3: Cleaning up Policy Engine...")
try:
    policy_client = PolicyClient(region_name=config["region"])
    policy_client.cleanup_policy_engine(config["policy_engine_id"])
    print("  Policy Engine and all policies deleted")
except Exception as e:
    print(f"  Warning: {e}")

# -----------------------------------------------------------------------
# 4. Delete SNS / SQS review infrastructure
# -----------------------------------------------------------------------
print("\nStep 4: Cleaning up review infrastructure...")
review_infra = config.get("review_infra", {})
if review_infra.get("sns_topic_arn") and review_infra.get("sqs_queue_url"):
    cleanup_review_infrastructure(
        region=config["region"],
        sns_topic_arn=review_infra["sns_topic_arn"],
        sqs_queue_url=review_infra["sqs_queue_url"],
    )
else:
    print("  No review infrastructure found in config")

# -----------------------------------------------------------------------
# 5. Delete Gateway
# -----------------------------------------------------------------------
print("\nStep 5: Cleaning up Gateway...")
try:
    gw_client.cleanup_gateway(
        config["gateway"]["gateway_id"],
        config["gateway"]["client_info"],
    )
    print("  Gateway and OAuth resources deleted")
except Exception as e:
    print(f"  Warning: {e}")

# -----------------------------------------------------------------------
# 6. Delete Lambda functions
# -----------------------------------------------------------------------
print("\nStep 6: Deleting Lambda functions...")
lambda_client = boto3.client("lambda", region_name=config["region"])
for name in config.get("lambdas", {}):
    try:
        lambda_client.delete_function(FunctionName=name)
        print(f"  Deleted: {name}")
    except Exception as e:
        print(f"  Warning deleting {name}: {e}")

# -----------------------------------------------------------------------
# 7. Delete IAM role
# -----------------------------------------------------------------------
print("\nStep 7: Cleaning up IAM role...")
iam_client = boto3.client("iam", region_name=config["region"])
role_name = "AgentCoreAutoPolicyLambdaRole"
try:
    iam_client.detach_role_policy(
        RoleName=role_name,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    iam_client.delete_role(RoleName=role_name)
    print(f"  IAM role '{role_name}' deleted")
except Exception as e:
    print(f"  Warning: {e}")

print("\n" + "=" * 70)
print("Cleanup complete!")
print("=" * 70)